# The Multivariate Gaussian

**Audience:** students who know the univariate normal and basic matrix algebra.

**What you will build:** the density of $\mathcal{N}(\boldsymbol{\mu},\Sigma)$,
linear transformations, contour plots for isotropic/correlated/heteroscedastic
covariances, marginals and conditionals, and Mahalanobis distance.

## Learning objectives

1. evaluate a bivariate Gaussian density from $\mu$ and $\Sigma$;
2. verify that affine maps preserve Gaussianity;
3. read covariance structure from contour ellipses;
4. compute conditional means/variances and Mahalanobis distances.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

rng = np.random.default_rng(0)
plt.rcParams.update(
    {
        "figure.figsize": (7, 5),
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
print("Setup complete")

## 1. Density and normalisation

$$p(\mathbf{x}) = \frac{1}{(2\pi)^{d/2}|\Sigma|^{1/2}}
\exp\!\left(-\tfrac12(\mathbf{x}-\mu)^\top\Sigma^{-1}(\mathbf{x}-\mu)\right).$$

The quadratic form in the exponent is the squared Mahalanobis distance. On a
fine grid the density should integrate approximately to one.

In [ ]:
mu = np.array([0.0, 0.0])
Sigma = np.array([[2.0, 0.6], [0.6, 1.0]])
dist = multivariate_normal(mean=mu, cov=Sigma)

grid = np.linspace(-6, 6, 400)
gx, gy = np.meshgrid(grid, grid)
points = np.column_stack([gx.ravel(), gy.ravel()])
pdf = dist.pdf(points).reshape(gx.shape)
cell_area = (grid[1] - grid[0]) ** 2
print(f"numerical integral = {pdf.sum() * cell_area:.5f}")
print(f"density at mean     = {dist.pdf(mu):.5f}")

## 2. Linear transformations

If $\mathbf{Z}\sim\mathcal{N}(0,I)$ and $\mathbf{X}=\mu+A\mathbf{Z}$, then
$\mathbf{X}\sim\mathcal{N}(\mu,AA^\top)$. We verify this by simulation with
$\mu=(1,2)^\top$ and $A=\begin{bmatrix}2&0\\1&3\end{bmatrix}$.

In [ ]:
mu = np.array([1.0, 2.0])
A = np.array([[2.0, 0.0], [1.0, 3.0]])
Sigma = A @ A.T

Z = rng.normal(size=(50_000, 2))
X = Z @ A.T + mu
print("theoretical mean:\n", mu)
print("sample mean:     ", np.round(X.mean(axis=0), 4))
print("theoretical covariance:\n", Sigma)
print("sample covariance:\n", np.round(np.cov(X, rowvar=False), 4))

## 3. Reading covariance from contours

Equal-density ellipses are level sets of the Mahalanobis distance. A scalar
covariance gives circles, positive off-diagonal entries tilt ellipses along
$x_1=x_2$, and unequal variances stretch the axes.

In [ ]:
cases = [
    ("isotropic", np.eye(2)),
    ("correlated", np.array([[1.0, 0.8], [0.8, 1.0]])),
    ("unequal variances", np.diag([2.0, 1.0])),
]
grid = np.linspace(-4, 4, 300)
gx, gy = np.meshgrid(grid, grid)
pts = np.column_stack([gx.ravel(), gy.ravel()])

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (title, cov) in zip(axes, cases):
    density = multivariate_normal(mean=np.zeros(2), cov=cov).pdf(pts)
    ax.contour(gx, gy, density.reshape(gx.shape), levels=5, cmap="viridis")
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
fig.tight_layout()
plt.show()

## 4. Marginals and conditionals

Partition $\Sigma$ into blocks. Then
$X_1\sim\mathcal{N}(\mu_1,\Sigma_{11})$ and
$X_2\mid X_1=x_1\sim\mathcal{N}(\mu_2+\Sigma_{21}\Sigma_{11}^{-1}(x_1-\mu_1),
\Sigma_{22}-\Sigma_{21}\Sigma_{11}^{-1}\Sigma_{12})$.
Conditioning shifts the mean and never increases the variance.

In [ ]:
mu = np.zeros(2)
Sigma = np.array([[2.0, 0.6], [0.6, 1.0]])
x1 = 1.0
cond_mean = mu[1] + Sigma[1, 0] / Sigma[0, 0] * (x1 - mu[0])
cond_var = Sigma[1, 1] - Sigma[1, 0] ** 2 / Sigma[0, 0]
print(f"X2 | X1={x1} ~ N({cond_mean:.3f}, {cond_var:.3f})")

xs = np.linspace(-4, 4, 300)
marginal = multivariate_normal(mu[1], Sigma[1, 1]).pdf(xs)
conditional = multivariate_normal(cond_mean, cond_var).pdf(xs)
fig, ax = plt.subplots()
ax.plot(xs, marginal, label="marginal X2")
ax.plot(xs, conditional, label=f"conditional X2 | X1={x1}")
ax.set(xlabel="x2", ylabel="density", title="Conditioning narrows the density")
ax.legend()
plt.show()

## 5. Mahalanobis distance

$$d_M(\mathbf{x},\mu)=\sqrt{(\mathbf{x}-\mu)^\top\Sigma^{-1}(\mathbf{x}-\mu)}.$$
It measures distance in standard-deviation units along each principal axis,
so correlations and unequal scales are handled correctly.

In [ ]:
def mahalanobis(x, mean, cov):
    diff = np.asarray(x) - np.asarray(mean)
    return float(np.sqrt(diff @ np.linalg.solve(cov, diff)))


mean = np.zeros(2)
cov = np.array([[2.0, 0.6], [0.6, 1.0]])
for point in [(0, 0), (2, 0), (1, 1), (-1, 1)]:
    print(f"d_M({point}) = {mahalanobis(point, mean, cov):.3f}")

## Exercises

1. Change `A` so that the sample covariance has a negative off-diagonal entry.
2. Verify that conditioning reduces the variance by the squared correlation.
3. Plot 50 samples and colour them by Mahalanobis distance.
4. Compare Euclidean and Mahalanobis nearest neighbours for the correlated covariance.
5. Use `np.linalg.cholesky` to generate samples and compare with `rng.multivariate_normal`.